## Step 1: Load cleaned data and import libraries

In [4]:
import numpy as np
import pandas as pd
df = pd.read_csv('../data/processed/cleaned_netflix.csv', parse_dates=['date_added'])
print(f"Shape: {df.shape}")
df.dtypes

Shape: (8807, 14)


show_id                     object
type                        object
title                       object
director                    object
cast                        object
country                     object
date_added          datetime64[ns]
release_year                 int64
rating                      object
duration                    object
listed_in                   object
description                 object
duration_minutes           float64
duration_seasons           float64
dtype: object

## Step 2: Split date_added into simpler columns

In [ ]:
# Split date_added
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month
df['month_name_added'] = df['date_added'].dt.strftime('%B')

# Decade from release_year
df['decade'] = (df['release_year']//10*10).astype('Int64')

#days_to_add: interval from release to time when netflix add Movie
df['days_to_add'] = (
    df['date_added'] - pd.to_datetime(df['release_year'].astype(str) + '-01-01')
).dt.days

# verify
df[['title', 'type', 'date_added', 'year_added', 'month_added', 'decade', 'days_to_add']].head(5)

,title,type,date_added,year_added,month_added,decade,days_to_add
0,Dick Johnson Is Dead,Movie,2021-09-25,2021.0,9.0,2020,633.0
1,Blood & Water,TV Show,2021-09-24,2021.0,9.0,2020,266.0
2,Ganglands,TV Show,2021-09-24,2021.0,9.0,2020,266.0
3,Jailbirds New Orleans,TV Show,2021-09-24,2021.0,9.0,2020,266.0
4,Kota Factory,TV Show,2021-09-24,2021.0,9.0,2020,266.0


## Step 3: Split multi-value columns to list

In [15]:
def split_to_list (series: pd.Series, delimiter: str = ',') -> pd.Series:
    return series.apply(
        lambda x: [item.strip() for item in x.split(delimiter)] if pd.notna(x) else []
    )

df['cast_list'] = split_to_list(df['cast'])
df['country_list'] = split_to_list(df['country'])
df['genre_list'] = split_to_list(df['listed_in'])
df['director_list'] = split_to_list(df['director'])

# Verify
print(df.loc[0, 'listed_in'])
print(df.loc[0,'genre_list'])

Documentaries
['Documentaries']


## Step 4: Genre flags and advance

### 1. Genre flags (boolean columns)

In [16]:
# get all genres in dataset
all_genres = df['genre_list'].explode().dropna().unique()
print(f"Numbers of unique genres: {len(all_genres)}")
print(sorted(all_genres))

Numbers of unique genres: 42
['Action & Adventure', 'Anime Features', 'Anime Series', 'British TV Shows', 'Children & Family Movies', 'Classic & Cult TV', 'Classic Movies', 'Comedies', 'Crime TV Shows', 'Cult Movies', 'Documentaries', 'Docuseries', 'Dramas', 'Faith & Spirituality', 'Horror Movies', 'Independent Movies', 'International Movies', 'International TV Shows', "Kids' TV", 'Korean TV Shows', 'LGBTQ Movies', 'Movies', 'Music & Musicals', 'Reality TV', 'Romantic Movies', 'Romantic TV Shows', 'Sci-Fi & Fantasy', 'Science & Nature TV', 'Spanish-Language TV Shows', 'Sports Movies', 'Stand-Up Comedy', 'Stand-Up Comedy & Talk Shows', 'TV Action & Adventure', 'TV Comedies', 'TV Dramas', 'TV Horror', 'TV Mysteries', 'TV Sci-Fi & Fantasy', 'TV Shows', 'TV Thrillers', 'Teen TV Shows', 'Thrillers']


In [17]:
genre_counts = (
    df['genre_list']
    .explode()
    .value_counts()
    .reset_index()
)
genre_counts.columns = ['genre', 'count']
genre_counts['pct'] = (genre_counts['count'] / len(df) *100).round(1)
genre_counts

,genre,count,pct
0,International Movies,2752,31.2
1,Dramas,2427,27.6
2,Comedies,1674,19.0
3,International TV Shows,1351,15.3
4,Documentaries,869,9.9
5,Action & Adventure,859,9.8
6,TV Dramas,763,8.7
7,Independent Movies,756,8.6
8,Children & Family Movies,641,7.3
9,Romantic Movies,616,7.0


In [20]:
# 13 selected genre flags
selected_genres ={
    # Common movie genres (>5%)
    'is_international' : 'International Movies',
    'is_drama' : 'Dramas',
    'is_comedy' : 'Comedies',
    'is_documentary' : 'Documentaries',
    'is_action' : 'Action & Adventure',
    'is_kids' : "Kids' TV",
    'is_romance' : 'Romantic Movies',
    'is_thriller' : 'Thrillers',
    'is_horror' : 'Horror Movies',
    # Typical TV genres
    'is_crime_tv' : 'Crime TV Shows',
    'is_reality_tv' : 'Reality TV',
    'is_standup' : 'Stand-Up Comedy',
    # Typical Region
    'is_anime' : 'Anime Series',

}

for flag_col, genre_name in selected_genres.items():
    df[flag_col] = df['genre_list'].apply(
        lambda genres: 1 if genre_name in genres else 0
    )

#Verify
flag_cols = list(selected_genres.keys())
df[flag_cols].sum().sort_values(ascending=False)

is_international    2752
is_drama            2427
is_comedy           1674
is_documentary       869
is_action            859
is_romance           616
is_thriller          577
is_crime_tv          470
is_kids              451
is_horror            357
is_standup           343
is_reality_tv        255
is_anime             176
dtype: int64

### 2. Content age (classify new/old content)

In [21]:
df['content_age'] = pd.cut(
    df['release_year'],
    bins=[0, 1980, 2000, 2010, 2015, 2021],
    labels=['Classic (<1980)', 'Vintage (1980-200)',
            'Modern (2000-2010)', 'Recent (2010-2015)',
            'Latest (2015+)']
)
df['content_age'].value_counts()

content_age
Latest (2015+)        5656
Recent (2010-2015)    1622
Modern (2000-2010)     967
Vintage (1980-200)     429
Classic (<1980)        133
Name: count, dtype: int64

### 3. Movie length category

In [24]:
df['movie_length_cat'] = pd.cut(
    df['duration_minutes'],
    bins=[0, 60, 90, 120, float('inf')],
    labels=['Short (<60min)', 'Standard (60-90min)',
            'Long (90-120min)', 'Extended (>120min)']
)
df[df['type'] == 'Movie']['movie_length_cat'].value_counts()

movie_length_cat
Long (90-120min)       2996
Standard (60-90min)    1506
Extended (>120min)     1142
Short (<60min)          487
Name: count, dtype: int64

### 4. Primary country and primary director (the first value from multi-value)

In [25]:
# Very useful when needing a value as a representative to group and visualize
# instead of exploding entire list
df['primary_country'] = df['country_list'].apply(lambda x: x[0] if x else None)
df['primary_director'] = df['director_list'].apply(lambda x: x[0] if x else None)

df[['title', 'country', 'primary_country', 'director', 'primary_director']].head(5)



,title,country,primary_country,director,primary_director
0,Dick Johnson Is Dead,United States,United States,Kirsten Johnson,Kirsten Johnson
1,Blood & Water,South Africa,South Africa,NaN,None
2,Ganglands,NaN,None,Julien Leclercq,Julien Leclercq
3,Jailbirds New Orleans,NaN,None,NaN,None
4,Kota Factory,India,India,NaN,None


## Step 5: Verify and export

In [26]:
print(f"Shape after feature engineeing: {df.shape}")
print(f'\nAdded columns:')
new_cols = [
    'year_added', 'month_added', 'month_name_added', 'decade',
    'days_to_add', 'cast_list', 'country_list', 'genre_list',
    'director_list', 'content_age', 'movie_length_cat',
    'primary_country', 'primary_director'
]
print(new_cols)
df[new_cols].head(3)

Shape after feature engineeing: (8807, 40)

Added columns:
['year_added', 'month_added', 'month_name_added', 'decade', 'days_to_add', 'cast_list', 'country_list', 'genre_list', 'director_list', 'content_age', 'movie_length_cat', 'primary_country', 'primary_director']


,year_added,month_added,month_name_added,decade,days_to_add,cast_list,country_list,genre_list,director_list,content_age,movie_length_cat,primary_country,primary_director
0,2021.0,9.0,September,2020,633.0,[],[United States],[Documentaries],[Kirsten Johnson],Latest (2015+),Standard (60-90min),United States,Kirsten Johnson
1,2021.0,9.0,September,2020,266.0,"[Ama Qamata, Khosi Ngema, Gail Mabalane, Thaba...",[South Africa],"[International TV Shows, TV Dramas, TV Mysteries]",[],Latest (2015+),NaN,South Africa,None
2,2021.0,9.0,September,2020,266.0,"[Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nab...",[],"[Crime TV Shows, International TV Shows, TV Ac...",[Julien Leclercq],Latest (2015+),NaN,None,Julien Leclercq


In [27]:
df.to_csv('../data/processed/featured_netflix.csv', index=False)
print(f"Exported: {df.shape[0]} rows, {df.shape[1]} columns")

Exported: 8807 rows, 40 columns
